# 8 · Gold, and the thing that notices when a number stops making sense

Two ideas in one notebook.

**Gold** is the answer, aggregated to the grain somebody actually asks for.
**The signal board** is what tells you when that answer stops being trustworthy.

| | |
|---|---|
| **reads** | `teach.silver_rides` |
| **writes** | `teach.gold_daily`, one row per day |

In [ ]:
import sys; sys.path.insert(0, '.')
from nb import show, sql, fetch, run, counts

import psycopg
from pipelines.lib.config import dsn, SCHEMA

print('silver rides:', f"{fetch(f'SELECT count(*) AS n FROM {SCHEMA}.silver_rides').n[0]:,}")

---

## Step 1 · The test for a gold column

> **If you cannot say the column name out loud in a sentence to a finance
> manager, it does not belong in gold.**

`avg_duration_min` passes. `duration_s` does not. `rides` passes. `trip_id`
does not, because a gold table is not about one ride.

In [ ]:
DDL = f"""
CREATE TABLE IF NOT EXISTS {SCHEMA}.gold_daily (
    trip_date        DATE PRIMARY KEY,
    rides            BIGINT,
    completed        BIGINT,
    cancelled        BIGINT,
    avg_distance_km  NUMERIC(8,2),
    avg_duration_min NUMERIC(8,1),
    avg_surge        NUMERIC(6,3),
    revenue          NUMERIC(14,2),
    settled_rides    BIGINT,
    unsettled_rides  BIGINT
);
"""

with psycopg.connect(dsn(), autocommit=True) as c:
    c.execute(DDL)

print('table ready')

---

## Step 2 · Aggregate with FILTER, not with three queries

`count(*) FILTER (WHERE ...)` counts only the rows matching a condition, **in
the same pass as everything else**. The alternative is three queries joined
together, which is slower and much harder to read.

In [ ]:
BUILD = f"""
SELECT
    trip_date,
    count(*)                                          AS rides,
    count(*) FILTER (WHERE status = 'completed')      AS completed,
    count(*) FILTER (WHERE status LIKE 'cancelled%%') AS cancelled,
    round(avg(distance_km), 2)                        AS avg_distance_km,
    round(avg(duration_min), 1)                       AS avg_duration_min,
    round(avg(surge), 3)                              AS avg_surge,
    round(coalesce(sum(fare), 0), 2)                  AS revenue,
    count(*) FILTER (WHERE is_settled)                AS settled_rides,
    count(*) FILTER (WHERE NOT is_settled)            AS unsettled_rides
FROM {SCHEMA}.silver_rides
GROUP BY trip_date
"""

with psycopg.connect(dsn(), autocommit=False) as c, c.cursor() as cur:
    rows_in = cur.execute(f'SELECT count(*) FROM {SCHEMA}.silver_rides').fetchone()[0]
    cur.execute(f'DELETE FROM {SCHEMA}.gold_daily')
    cur.execute(f'INSERT INTO {SCHEMA}.gold_daily {BUILD}')
    rows_out = cur.rowcount
    c.commit()

print(f'{rows_in:,} rides  ->  {rows_out} days')

In [ ]:
sql(f"""
    SELECT trip_date, rides, completed, cancelled, avg_distance_km,
           avg_duration_min, avg_surge, revenue, settled_rides, unsettled_rides
    FROM {SCHEMA}.gold_daily ORDER BY trip_date DESC
""", 'gold_daily, the whole table')

### Two details in that query worth a minute

**`avg()` ignores NULLs by itself.** `avg_surge` is therefore the average across
rides that *have* a surge value, not across all rides with the missing ones
counted as zero.

That is what you want, and it has a consequence: when a field goes missing
upstream, this number does **not** drag downwards. The row count behind it drops
instead, which is a completely different signal. You will watch exactly that
happen in notebook 9.

**`coalesce(sum(fare), 0)`** because `sum()` of nothing is `NULL`, not zero, and
a day with no fares should report `0` revenue rather than an empty cell.

---

## Step 3 · Rebuild it, for the same reason as silver

Thirty rows. A full rebuild takes milliseconds. Anything cleverer would be
complexity nobody is paying for.

In [ ]:
run('-m', 'pipelines.p8_gold_daily')

---

# Now the second half. Who tells you when this is wrong?

You have eight pipelines. They all say `ok`. **That is not the same as the
numbers being right.**

Three completely different things can check your work, and people collapse them
into one word, quality, and then argue past each other.

![](img/gold-1-checkpoints.png)

| | the contract | the tests | the signal board |
|---|---|---|---|
| **where** | inside the pipeline | inside the transaction | outside everything |
| **when** | per record, on the way in | after the write, before the commit | on a clock, after the run |
| **can it block?** | **yes** | **yes**, it rolls back | **no. Nothing. Ever.** |

> **A check that can block a write is a contract. A check that cannot is a
> signal.**

You have already built the first one, eight times over. This is the third.

---

## Step 4 · A signal is four things

![](img/gold-2-signal.png)

Build one, here, in a cell.

In [ ]:
from dataclasses import dataclass

@dataclass
class Signal:
    name: str            # so a person can talk about it
    owner: str           # who gets told when it moves
    sql: str             # returns exactly one number
    baseline: float      # what that number normally is
    tolerance: float = 0.0
    means: str = ''

held = Signal(
    name='records_held',
    owner='data-platform',
    sql=f'SELECT count(*) FROM {SCHEMA}.quarantine',
    baseline=0,
    means='Records a contract refused. Not lost, not wrong, just not published.')

with psycopg.connect(dsn()) as c:
    value = float(c.execute(held.sql).fetchone()[0])

gap = abs(value - held.baseline)
breached = gap > max(held.tolerance, 0.0001)

print(f'{held.name:16} = {value:,.0f}   baseline {held.baseline}   '
      f'{"BREACH" if breached else "ok"}')
if breached:
    print(f'  -> this one is {held.owner}\'s. {held.means}')

**That is the whole idea.** There is no model here and no intelligence.

Deciding that 34% is far from 0.09% is **arithmetic**, and pretending otherwise
is how people end up buying something they could have written in an afternoon.

### The field that actually matters is `owner`

A breach with no name attached is a number on a screen that everybody assumes
somebody else is looking at.

---

## Step 5 · Two kinds of baseline

![](img/gold-3-baselines.png)

Some numbers have **a history**. Some have **a rule**.

In [ ]:
import statistics

# a baseline computed from history: what have the other days looked like?
with psycopg.connect(dsn()) as c:
    today   = float(c.execute(
        f'SELECT rides FROM {SCHEMA}.gold_daily ORDER BY trip_date DESC LIMIT 1').fetchone()[0])
    history = [float(r[0]) for r in c.execute(
        f'SELECT rides FROM {SCHEMA}.gold_daily ORDER BY trip_date DESC OFFSET 1')]

baseline = statistics.mean(history)
spread   = max(statistics.pstdev(history), 0.01)
z        = (today - baseline) / spread

print(f'most recent day : {today:>10,.0f} rides')
print(f'the days before : {baseline:>10,.0f} on average, spread {spread:,.0f}')
print(f'z score         : {z:>10.2f}   (how many spreads away from normal)')
print()
print('BREACH' if abs(z) > 4 else 'ok, within normal variation')

### Why 4 and not 2

A z score of 2 fires on a quiet Tuesday. A z score of 4 fires when something
genuinely broke.

Set it too tight and the board cries wolf, people stop reading it, and you have
spent effort building something that made you *less* safe than having nothing.

### And one signal needs both kinds

`events_per_ride` has a **fixed** baseline of `4.7` with a **tolerance** of
`1.0`. A completed ride emits five events and a cancelled one emits three, so
the true average sits a little under five, and that is correct, not a fault.

Set that baseline to exactly `5` and the board breaches on day one, for a
perfectly healthy system. That happened while building this course.

---

## Step 6 · The whole board

Six signals. Between them they answer the four questions that go wrong: **did it
arrive, is it the right amount, is it the right shape, and does it still add
up.**

In [ ]:
from signals.board import SIGNALS

for s in SIGNALS:
    kind = ('fixed at %g' % s.fixed_baseline) if s.fixed_baseline is not None else 'from history'
    print(f'  {s.name:22} {s.owner:16} {kind}')
    print(f'  {"":22} watches {s.watches}')
    print()

In [ ]:
run('-m', 'signals.board')

### Read the one that is not about data at all

`pipelines_failing` looks at `teach.runs`, not at any table of records. It asks
**whether the work happened.**

That matters because **a pipeline that never ran leaves perfectly valid data
behind.** Every row is correct. Every row is old. No check that looks only at
values will ever notice.

---

## Step 7 · Where the board runs, and where it must not

**Beside the warehouse, on a clock, after the pipelines have finished.**

Never inside a pipeline. A check that can block a write is a different thing
with a different name, and you already built that one: it is the contract.

There is a dashboard for the same six signals:

```bash
python cli.py dashboard        #  http://localhost:8099
```

Put it on the second screen and leave it there.

---

## What you learned

- Gold is **the answer at the grain somebody asks for**. If you cannot say the
  column name to a finance manager, it does not belong
- `FILTER (WHERE ...)` beats three joined queries
- `avg()` ignores nulls, so a missing field moves **the count**, not the average
- **Three checkpoints**: contract, tests, signal board. Only the first two can block
- A signal is **a name, a query, a baseline, and an owner**. The owner is the point
- **Two kinds of baseline**: computed from history, or decided by a person
- A threshold that is too tight makes you **less** safe, because people stop reading
- `pipelines_failing` is the signal that catches the failure no data check can see